# SRT4U — Benchmark analysis

Reproducible analysis of a `TranslationBenchmark` export. Loads
`analysis/sample_benchmark.json` (synthetic stub-provider data committed
for reproducibility), validates it, and produces descriptive stats,
variability, error/fallback, size-relationship and data-quality analysis.

Requirements (analysis env only, **not** runtime deps): `pandas`,
`matplotlib`. Point `REPORT` at any `benchmark/*.json` export to analyze
real runs. No network, no translation logic here.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

from analysis.benchmark_analysis import (
    clean_runs,
    data_quality,
    describe,
    error_summary,
    load_report,
    run_analysis,
    size_relationship,
    to_dataframe,
    validate_runs,
)

REPORT = "analysis/sample_benchmark.json"
OUT_DIR = "/tmp/srt4u-nbcheck"

## 1. Load & validate — how are problematic rows treated?

`clean_runs` coerces CSV-style strings to typed values, keeps failed
executions as data, and reports every anomaly in `notes` instead of
dropping rows silently.

In [ ]:
report = load_report(REPORT)
print("benchmark_id:", report["benchmark_id"])
print("dataset:", report["dataset_version"], report["dataset_size"], "cues")
print("config:", report["config"])
runs, notes = clean_runs(report["runs"])
print("cleaned rows:", len(runs))
print("notes:", notes)
validation = validate_runs(runs)
print("failed runs:", validation["failed_runs"])
print("fallback runs:", validation["fallback_runs"])
print("impossible values:", validation["impossible_values"] or "none")

## 2. Descriptive stats per provider (successful runs)

In [ ]:
df = to_dataframe(runs)
metrics = ["duration_ms", "latency_per_cue_ms", "characters_per_second"]
ok = df[df["success"] == True]  # noqa: E712
print(ok.groupby("requested_provider")[metrics].agg(["count", "mean", "median", "min", "max", "std"]))
for entry in describe(runs):
    print(entry["group"], "n =", entry["n_total"],
          "median latency/cue =", round(entry["latency_per_cue_ms.median"], 2))

## 3. Variability — never the mean alone (std, IQR, p10–p90, CV)

In [ ]:
for entry in describe(runs):
    print(entry["group"],
          "stdev =", round(entry["latency_per_cue_ms.stdev"] or 0, 2),
          "IQR =", round(entry["latency_per_cue_ms.iqr"] or 0, 2),
          "p10–p90 =", (round(entry["latency_per_cue_ms.p10"] or 0, 1),
                         round(entry["latency_per_cue_ms.p90"] or 0, 1)),
          "CV =", round(entry["latency_per_cue_ms.cv"] or 0, 3))

## 4. Errors & fallback by provider (rates, not rankings)

In [ ]:
for entry in error_summary(runs):
    print(entry["group"],
          f"success={entry['success_rate']:.0%}",
          f"fallback={entry['fallback_rate']:.0%}",
          "errors =", entry["error_types"])

## 5. Size relationship — does input size drive duration?

In [ ]:
sizes = size_relationship(runs)
print("n_successful:", sizes["n_successful"])
for name, value in sizes["pearson"].items():
    print(f"{name}: {value:.3f}" if value is not None else f"{name}: n/a")

## 6. Data quality & limits

In [ ]:
for finding in data_quality(runs):
    print(f"{finding['check']}: {finding['count']} — {finding['detail']}")

## 7. Visualizations — one question per chart

In [ ]:
import pathlib

plots = pathlib.Path(OUT_DIR) / "nb_plots"
plots.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots()  # Q: how is latency distributed?
ax.hist(ok["latency_per_cue_ms"], bins=10)
ax.set_title("Latency per cue distribution")
ax.set_xlabel("latency_per_cue_ms"); ax.set_ylabel("executions")
fig.tight_layout(); fig.savefig(plots / "nb_latency_hist.png")

fig, ax = plt.subplots()  # Q: which provider is faster / more variable?
ok.boxplot(column="latency_per_cue_ms", by="requested_provider", ax=ax)
ax.set_title("Latency per cue by provider")
fig.tight_layout(); fig.savefig(plots / "nb_latency_boxplot.png")

fig, ax = plt.subplots()  # Q: does duration grow with input size?
for provider, group in ok.groupby("requested_provider"):
    ax.scatter(group["characters_input"], group["duration_ms"], label=provider)
ax.set_title("Duration vs input characters")
ax.set_xlabel("characters_input"); ax.set_ylabel("duration_ms")
ax.legend(); fig.tight_layout(); fig.savefig(plots / "nb_latency_vs_chars.png")
print("plots in", plots)

## 8. Full pipeline export — `summary.csv`, `cleaned_dataset.csv`, `analysis_report.json`, `plots/`

In [ ]:
payload = run_analysis(report, OUT_DIR)
print("artifacts:", sorted(str(p) for p in pathlib.Path(OUT_DIR).rglob('*') if p.is_file()))
print("\nconclusions:")
for line in payload["conclusions"]:
    print("-", line)